# Bitcoin Trader Performance vs Market Sentiment Analysis
### Hyperliquid Historical Trades × Fear & Greed Index
**Dataset:** 211,224 trades | Dec 2023 – Apr 2025 | 105 unique traders

## 1. Setup & Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

print("Libraries loaded successfully")

## 2. Load & Merge Datasets

In [ ]:
trades = pd.read_csv('historical_data.csv')
fg = pd.read_csv('fear_greed_index.csv')

# Parse dates
trades['date'] = pd.to_datetime(trades['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.date
fg['date'] = pd.to_datetime(fg['date']).dt.date

# Merge on date
df = trades.merge(fg[['date', 'value', 'classification']], on='date', how='left')

print(f"Total trades: {len(df):,}")
print(f"Unique traders: {df['Account'].nunique()}")
print(f"Unique coins: {df['Coin'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Sentiment coverage: {df['classification'].notna().sum():,} trades matched")
print()
print("Sentiment distribution:")
print(df['classification'].value_counts())
df.head()

## 3. PnL Analysis by Market Sentiment

In [ ]:
sent_order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
colors = ['#c0392b', '#e74c3c', '#95a5a6', '#27ae60', '#1a6b3a']

pnl_sent = df.groupby('classification')['Closed PnL'].agg(['sum','mean','count']).reindex(sent_order)
pnl_sent.columns = ['Total PnL ($)', 'Avg PnL per Trade ($)', 'Trade Count']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].bar(sent_order, pnl_sent['Total PnL ($)'], color=colors)
axes[0].set_title('Total PnL by Sentiment', fontweight='bold')
axes[0].set_ylabel('USD')
axes[0].tick_params(axis='x', rotation=30)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\${x/1e6:.1f}M'))

axes[1].bar(sent_order, pnl_sent['Avg PnL per Trade ($)'], color=colors)
axes[1].set_title('Avg PnL per Trade', fontweight='bold')
axes[1].set_ylabel('USD')
axes[1].tick_params(axis='x', rotation=30)

axes[2].bar(sent_order, pnl_sent['Trade Count'], color=colors)
axes[2].set_title('Trade Volume by Sentiment', fontweight='bold')
axes[2].set_ylabel('Number of Trades')
axes[2].tick_params(axis='x', rotation=30)

plt.suptitle('PnL Analysis by Market Sentiment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart1_pnl_by_sentiment.png', bbox_inches='tight')
plt.show()

print(pnl_sent.to_string())

## 4. Win Rate by Sentiment

In [ ]:
df['is_win'] = df['Closed PnL'] > 0
closed = df[df['Closed PnL'] != 0]
win_rate = closed.groupby('classification')['is_win'].mean().reindex(sent_order) * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(sent_order, win_rate, color=colors)
ax.set_ylim(60, 95)
ax.set_title('Win Rate by Market Sentiment (Closed Trades Only)', fontweight='bold', fontsize=13)
ax.set_ylabel('Win Rate (%)')
ax.tick_params(axis='x', rotation=15)

for bar, val in zip(bars, win_rate):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{val:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('chart2_win_rate.png', bbox_inches='tight')
plt.show()

print("Win Rates:")
for s, w in zip(sent_order, win_rate):
    print(f"  {s}: {w:.1f}%")

## 5. Long vs Short Directional Bias by Sentiment

In [ ]:
bias = df[df['Direction'].isin(['Open Long', 'Open Short'])]       .groupby(['classification', 'Direction']).size().unstack(fill_value=0).reindex(sent_order)

x = np.arange(len(sent_order))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w/2, bias['Open Long'], w, label='Open Long', color='#2980b9')
ax.bar(x + w/2, bias['Open Short'], w, label='Open Short', color='#c0392b')
ax.set_xticks(x)
ax.set_xticklabels(sent_order)
ax.set_title('Long vs Short Position Openings by Market Sentiment', fontweight='bold', fontsize=13)
ax.set_ylabel('Number of Trades')
ax.legend()

plt.tight_layout()
plt.savefig('chart3_long_short_bias.png', bbox_inches='tight')
plt.show()

bias['Long/Short Ratio'] = (bias['Open Long'] / bias['Open Short']).round(2)
print(bias)

## 6. Monthly PnL vs Fear/Greed Index Score

In [ ]:
df['month'] = pd.to_datetime(df['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.to_period('M')
monthly = df.groupby('month').agg(
    Total_PnL=('Closed PnL', 'sum'),
    Avg_FG=('value', 'mean'),
    Trade_Count=('Closed PnL', 'count')
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)

fig, ax1 = plt.subplots(figsize=(16, 6))
bar_colors = ['#27ae60' if v >= 0 else '#c0392b' for v in monthly['Total_PnL']]
ax1.bar(monthly['month_str'], monthly['Total_PnL'], color=bar_colors, alpha=0.85, label='Monthly PnL')
ax1.set_ylabel('Total PnL (USD)', fontsize=12)
ax1.tick_params(axis='x', rotation=45)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\${x/1e6:.1f}M'))

ax2 = ax1.twinx()
ax2.plot(monthly['month_str'], monthly['Avg_FG'], color='orange', linewidth=2.5,
         marker='o', markersize=5, label='Avg Fear/Greed Score')
ax2.set_ylabel('Avg Fear/Greed Score (0-100)', color='orange', fontsize=12)
ax2.set_ylim(0, 110)

ax1.set_title('Monthly PnL vs Fear/Greed Index Score (Dec 2023 – Apr 2025)',
              fontweight='bold', fontsize=13)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.tight_layout()
plt.savefig('chart4_monthly_pnl_fg.png', bbox_inches='tight')
plt.show()

print(monthly[['month_str','Total_PnL','Avg_FG','Trade_Count']].to_string(index=False))

## 7. Top Coins & Top Traders

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_coins = df.groupby('Coin')['Closed PnL'].sum().sort_values(ascending=False).head(10)
axes[0].barh(top_coins.index[::-1], top_coins.values[::-1], color='#2980b9')
axes[0].set_title('Top 10 Coins by Total PnL', fontweight='bold')
axes[0].set_xlabel('Total PnL (USD)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\${x/1e6:.1f}M'))

top_traders = df.groupby('Account')['Closed PnL'].sum().sort_values(ascending=False).head(10)
short_labels = [a[:8]+'...' for a in top_traders.index]
axes[1].barh(short_labels[::-1], top_traders.values[::-1], color='#8e44ad')
axes[1].set_title('Top 10 Traders by Total PnL', fontweight='bold')
axes[1].set_xlabel('Total PnL (USD)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'\${x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig('chart5_top_coins_traders.png', bbox_inches='tight')
plt.show()

print("Top 5 Coins:")
print(top_coins.head().to_string())

## 8. Leverage Impact Analysis

In [ ]:
lev_stats = df.groupby('Crossed')['Closed PnL'].agg(['mean','sum','count'])
lev_stats.index = ['Isolated', 'Crossed (Leveraged)']
lev_stats.columns = ['Avg PnL', 'Total PnL', 'Trade Count']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['Isolated\n(not crossed)', 'Crossed\n(leveraged)'],
              [lev_stats.loc['Isolated','Avg PnL'], lev_stats.loc['Crossed (Leveraged)','Avg PnL']],
              color=['#27ae60', '#c0392b'], width=0.4)
ax.set_title('Avg PnL per Trade: Isolated vs Leveraged', fontweight='bold', fontsize=13)
ax.set_ylabel('Avg PnL (USD)')
for bar, val in zip(bars, [lev_stats.loc['Isolated','Avg PnL'], lev_stats.loc['Crossed (Leveraged)','Avg PnL']]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'\${val:.1f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('chart6_leverage_analysis.png', bbox_inches='tight')
plt.show()

print(lev_stats.to_string())

## 9. Key Insights & Trading Strategy Recommendations

---

### Finding 1: Fear Periods = Best Buying Opportunity
- Fear periods generated the **highest total PnL ($3.36M)** across the dataset
- Traders opened **1.64× more longs than shorts** during Fear — confirming "buy the fear" behavior
- **Strategy Recommendation:** Accumulate long positions when Fear & Greed Index < 30

---

### Finding 2: Extreme Greed = Highest Win Rate
- Extreme Greed had the **highest win rate (89.2%)** and **best avg PnL per trade ($68)**
- Momentum-following strategies perform best in euphoric market conditions
- **Strategy Recommendation:** Ride momentum trends during Extreme Greed (F&G > 75) but set tight stop-losses

---

### Finding 3: Contrarian Shorting in Greed Zones
- During Greed, short openings **outnumbered longs 1.36×**
- Experienced traders fade rallies and position against retail euphoria at market tops
- **Strategy Recommendation:** Scale into short positions when F&G > 75 with clear invalidation levels

---

### Finding 4: Leverage Significantly Hurts Returns
- Crossed (leveraged) trades averaged only **$35.6 PnL** vs **$69.1** for isolated — a **94% gap**
- Leverage amplifies losses during volatile reversals, destroying avg returns
- **Strategy Recommendation:** Prefer isolated margin or keep leverage below 3×

---

### Finding 5: December 2024 ATH Concentration Risk
- December 2024 alone generated **$3.0M (29% of all-time PnL)** during Bitcoin's ATH
- High PnL is extremely concentrated in a few months — most months are near breakeven
- **Strategy Recommendation:** Increase position sizing during confirmed bull runs with F&G > 70

---

### Finding 6: Altcoins Outperform BTC on Per-Trade Basis
- SOL avg PnL: **$153/trade** | ETH: **$118/trade** | BTC: only **$33/trade**
- Altcoin volatility creates better asymmetric trading opportunities
- **Strategy Recommendation:** Allocate a portion of portfolio to high-volatility altcoins during bull markets

---

## Summary Table

| Sentiment | Total PnL | Avg PnL/Trade | Win Rate | Best Strategy |
|-----------|-----------|---------------|----------|---------------|
| Extreme Fear | $739K | $35 | 76.2% | Buy & hold |
| Fear | $3.36M | $54 | 87.3% | Aggressive long |
| Neutral | $1.29M | $34 | 82.4% | Wait & watch |
| Greed | $2.15M | $43 | 76.9% | Short contrarian |
| Extreme Greed | $2.72M | $68 | 89.2% | Momentum long |
